# 🎬 ASTRAA — LTX-Video Colab
Free/open-source image-to-video test for ASTRAA.

## 1. Check GPU
Run this first. A free T4 is ideal for this lightweight test.

In [ ]:
!nvidia-smi


## 2. Install LTX-Video
Uses the official Lightricks repository.

In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git
!cd /content/LTX-Video && git fetch --depth 1 origin bdc8f017f0148a0f0bb9e3a5049d2d356423cee0 && git checkout bdc8f017f0148a0f0bb9e3a5049d2d356423cee0
%cd /content/LTX-Video
!pip install -q -e '.[inference]'
!pip install -q --force-reinstall --no-deps 'huggingface-hub~=0.30'
!pip install -q 'accelerate>=0.26'


## 3. Download the lighter 2B distilled model
This is intended for lighter VRAM than the 13B model.

In [ ]:
from huggingface_hub import hf_hub_download
model_dir='/content/LTX-Video/models'
import os
os.makedirs(model_dir, exist_ok=True)
hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=model_dir)
print('Model downloaded:', model_dir)


## 4. Upload the ASTRAA reference image
Upload one image such as Aarav + Maa Meera. Keep the image in `/content/LTX-Video/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
image_name=next(iter(uploaded))
image_path=f'/content/LTX-Video/{image_name}'
print(image_path)


## 5. Generate the first test shot
Start with an ultra-small 9-frame T4 test. This first run uses CPU offload to avoid a T4 runtime crash.


In [ ]:
import os, subprocess, yaml, glob, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE = "/content/LTX-Video"
MODULE = f"{BASE}/ltx_video/inference.py"
RUNNER = f"{BASE}/inference.py"
BASE_CONFIG = f"{BASE}/configs/ltxv-2b-0.9.8-distilled.yaml"
MODEL = f"{BASE}/models/ltxv-2b-0.9.8-distilled.safetensors"
LOCAL_CONFIG = f"{BASE}/configs/astraa-2b-t4-safe.yaml"
OUTPUT = f"{BASE}/outputs/astraa_test"

for path in (MODULE, RUNNER, BASE_CONFIG, MODEL):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing required file: {path}. Run Steps 2-4 first.")

src = open(MODULE, encoding="utf-8").read()

old_startup = """    transformer = transformer.to(device)
    vae = vae.to(device)
    text_encoder = text_encoder.to(device)
"""
new_startup = """    # Keep components on CPU during construction.
    # Diffusers CPU offload is enabled after the pipeline is created.
"""

if old_startup in src:
    src = src.replace(old_startup, new_startup, 1)
elif new_startup not in src:
    raise RuntimeError("LTX startup block is neither original nor already patched.")

old_pipeline = """    pipeline = LTXVideoPipeline(**submodel_dict)
    pipeline = pipeline.to(device)
    return pipeline
"""
new_pipeline = """    pipeline = LTXVideoPipeline(**submodel_dict)
    if device == "cuda":
        pipeline.enable_model_cpu_offload(device=device)
    else:
        pipeline = pipeline.to(device)
    return pipeline
"""

if old_pipeline in src:
    src = src.replace(old_pipeline, new_pipeline, 1)
elif new_pipeline not in src:
    raise RuntimeError("LTX pipeline block is neither original nor already patched.")

open(MODULE, "w", encoding="utf-8").write(src)

with open(BASE_CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["checkpoint_path"] = MODEL
cfg.pop("pipeline_type", None)
cfg.pop("spatial_upscaler_model_path", None)
cfg["prompt_enhancement_words_threshold"] = 0

with open(LOCAL_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

images = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    images.extend(glob.glob(f"{BASE}/{ext}"))

if not images:
    raise FileNotFoundError("No reference image found in /content/LTX-Video. Run Step 4 once.")

IMAGE = images[0]
os.makedirs(OUTPUT, exist_ok=True)

PROMPT = (
    "Aarav's mother Meera remains visually consistent with the reference image. "
    "She is inside her house at night. The curtains gently move in the wind. "
    "She slowly looks toward the entrance with a worried protective expression. "
    "Slow cinematic camera push-in, subtle dust particles, dramatic nighttime lighting, "
    "high-quality 3D animated Indian fantasy movie style, natural motion, no dialogue, no text."
)

print("IMAGE:", IMAGE)
print("MODEL:", os.path.isfile(MODEL))
print("CONFIG:", os.path.isfile(LOCAL_CONFIG))
print("RUNNER:", RUNNER)

result = subprocess.run(
    [
        sys.executable,
        RUNNER,
        "--prompt", PROMPT,
        "--output_path", OUTPUT,
        "--pipeline_config", LOCAL_CONFIG,
        "--conditioning_media_paths", IMAGE,
        "--conditioning_start_frames", "0",
        "--height", "128",
        "--width", "192",
        "--num_frames", "9",
        "--frame_rate", "24",
        "--seed", "42",
    ],
    cwd=BASE,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"LTX generation failed with exit code {result.returncode}")

videos = glob.glob(f"{OUTPUT}/*.mp4")
images_out = glob.glob(f"{OUTPUT}/*.png")
print("VIDEOS:", videos)
print("IMAGES:", images_out)

if not videos and not images_out:
    raise RuntimeError("LTX finished without creating an MP4 or PNG.")

print("SUCCESS: ASTRAA first test shot generated.")


In [ ]:
import os, glob
videos=glob.glob('/content/LTX-Video/**/*.mp4', recursive=True)
print('\n'.join(videos[-10:]) if videos else 'No MP4 found yet.')


## Next
Once the first shot works, we will add reusable ASTRAA prompts and an extension workflow.